In [ ]:
"""
سوال6

"""
# اضافه کردن کتابخانه ها 
import arabic_reshaper
import jdatetime
import matplotlib.pyplot as plt
import pandas as pd
import os

# از راست به چپ کردن متن فارسی
from bidi.algorithm import get_display
plt.rcParams['font.family'] = 'Tahoma'

def fix_fa(text):
    reshaped_text = arabic_reshaper.reshape(str(text))
    return get_display(reshaped_text)

# خواندن دیتای تمیز شده جدید
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop", "final_data.csv")
df_rent = pd.read_csv(desktop_path)

# اطمینان از اینکه ستون تاریخ به فرمت datetime تبدیل شده است (برای مرتب‌سازی صحیح)
df_rent['date_dt'] = pd.to_datetime(df_rent['created_at_month'])

# حذف داده‌های پرت و اشتباه (حذف اجاره‌های بالای ۵۰۰ میلیون تومان یا زیر ۱۰ هزار تومان)
df_rent = df_rent[(df_rent['rent_value'] >= 10000) & (df_rent['rent_value'] <= 500000000)]

# محاسبه میانه‌ اجاره هر ماه (استفاده از median به جای mean جهت جلوگیری از خراب شدن نمودار با داده‌های پرت)
trend_df = (df_rent
    .groupby(['date_dt', 'created_at_month'])['rent_value']
    .median()
    .reset_index())

# مرتب کردن تاریخ به ترتیب زمانی
trend_df = trend_df.sort_values('date_dt')

# تبدیل میلادی به شمسی
def to_jalali_month(dt):
    j_date = jdatetime.date.fromgregorian(date=dt.date())
    months_fa = [
        'فروردین', 'اردیبهشت', 'خرداد', 'تیر',
        'مرداد', 'شهریور', 'مهر', 'آبان',
        'آذر', 'دی', 'بهمن', 'اسفند'
    ]
    # متن مناسب نمایش در نمودار
    raw_str = f'{months_fa[j_date.month - 1]} {j_date.year}'
    return fix_fa(raw_str)

# ساختن ستون ماه شمسی 
trend_df['jalali_month'] = trend_df['date_dt'].apply(to_jalali_month)

# تعیین اندازه و رسم نمودار خطی 
plt.figure(figsize=(14, 6))
plt.plot(
    trend_df['jalali_month'],
    trend_df['rent_value'],
    marker='o',
    linestyle='-',
    color='#761314',
    linewidth=2.5
)

# تعیین عنوان نمودار
plt.title(
    fix_fa('ترند میانه قیمت اجاره بر حسب ماه‌های شمسی'),
    fontsize=15,
    pad=15
)

# اسامی محورها
plt.xlabel(fix_fa('ماه ثبت آگهی'), fontsize=12)
plt.ylabel(fix_fa('میانه قیمت اجاره (تومان)'), fontsize=12)

# مرتب کردن فاصله ها
plt.xticks(rotation=45, ha='right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

# نمایش
plt.show()

In [ ]:
"""
سوال7
"""
# وارد کردن کتابخانه ها
import os
import arabic_reshaper
from bidi.algorithm import get_display
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import pandas as pd


# متن فارسی و راست به چپ
def fix_fa(text):
    reshaped_text = arabic_reshaper.reshape(str(text))
    return get_display(reshaped_text)


# خواندن دیتای تمیز شده جدید
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop", "final_data.csv")
df = pd.read_csv(desktop_path, low_memory=False)

# تبدیل تاریخ به فرمت datetime برای استخراج سال و ماه
df['created_at_month'] = pd.to_datetime(df['created_at_month'], errors='coerce')


# تابع تبدیل تاریخ میلادی به سال شمسی
def gregorian_to_jalali_year(date):
    if pd.isna(date):
        return None

    year = date.year
    if year == 2024:
        start_date = pd.Timestamp('2024-03-20')
    else:
        start_date = pd.Timestamp(f'{year}-03-21')

    if date >= start_date:
        return year - 621
    else:
        return year - 622


# ساخت ستون year از روی تاریخ موجود در دیتای تمیز
df['year'] = df['created_at_month'].apply(gregorian_to_jalali_year)

# انتخاب سال های مورد نظر
df = df[df['year'].isin([1400, 1401, 1402, 1403])]

# نرخ تورم سالانه (داده ها از سایت رسمی بانک مرکزی)
inflation = {1400: 46.2, 1401: 53.1, 1402: 47.4, 1403: 35.8}

# شاخص قیمت (سال پایه 1400)
price_index = {1400: 100}
for year in [1401, 1402, 1403]:
  price_index[year] = price_index[year - 1] * (1 + inflation[year] / 100)

# میانگین قیمت هر سال
annual_price = df.groupby('year')['price_value'].mean().reset_index()

# گذاشتن قیمت در جدول
annual_price['price_index'] = annual_price['year'].map(price_index)

# محاسبه قیمت حقیقی ( سال پایه 1403)
base_year = 1403
annual_price['real_price'] = (
    annual_price['price_value'] * price_index[base_year]
    / annual_price['price_index'])


years = annual_price['year'].tolist()
year_labels = [fix_fa(f'سال {int(year)}') for year in years]


# رسم نمودار
plt.figure(figsize=(11, 6))
plt.plot(
    years,
    annual_price['real_price'],
    color='#4D0547',
    marker='o',
    markersize=8,
    linewidth=3.5,)

# فارسی کردن محور ایکس
plt.xticks(years, year_labels, fontsize=11)


# فارسی کردن اعداد محور ایگرگ
def format_y(value, position):
    return fix_fa(f'{value:,.0f}')


plt.gca().yaxis.set_major_formatter(FuncFormatter(format_y))

# عنوان محور ها
plt.xlabel(fix_fa('سال'), fontsize=13)
plt.ylabel(fix_fa('قیمت حقیقی'), fontsize=13)
plt.title(
    fix_fa('روند قیمت حقیقی ملک در سال‌های ۱۴۰۰ تا ۱۴۰۳'),
    fontsize=15,
    fontweight='bold',)
plt.grid(True,linestyle='--', alpha=0.8, linewidth=1.5)
# نمایش
plt.tight_layout()
plt.show()

In [ ]:
"""
سوال8
"""
# وارد کردن کتابخانه‌ها
import os
import arabic_reshaper
from bidi.algorithm import get_display
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


# نمایش فارسی و راست به چپ
def fix_fa(text):
  reshaped_text = arabic_reshaper.reshape(str(text))
  return get_display(reshaped_text)


# خواندن فایل تمیز شده جدید و ستون های مورد نیاز
file_path = os.path.join(os.path.expanduser("~"), "Desktop", "final_data.csv")
columns = [
    "price_value",
    "land_size",
    "building_size",
    "regular_person_capacity",
    "rooms_count",
    "location_latitude",
    "location_longitude",
]

df = pd.read_csv(file_path, usecols=columns)

# اطمینان از اینکه ستون‌های عددی به صورت float یا int هستند
for col in columns:
  df[col] = pd.to_numeric(df[col], errors="coerce")

# ماتریس همبستگی
correlation_matrix = df[columns].corr(method="pearson")

# ترجمه ستون ها 
persian_labels = {
    "price_value": "مبلغ قیمت",
    "land_size": "متراژ زمین",
    "building_size": "زیربنا",
    "regular_person_capacity": "ظرفیت نفرات",
    "rooms_count": "تعداد اتاق‌ها",
    "location_latitude": "عرض جغرافیایی",
    "location_longitude": "طول جغرافیایی",
}

labels = [fix_fa(persian_labels[col]) for col in columns]

# اعداد داخل نمودار
annot_matrix = correlation_matrix.map(
    lambda x: "N/A" if pd.isna(x) else f"{x:.2f}"
)

# رنگ
cmap = plt.cm.coolwarm.copy()
cmap.set_bad("#BDBDBDC7")

# ایجاد شکل
fig, ax = plt.subplots(figsize=(14, 12))
fig.subplots_adjust(top=0.90, bottom=0.20, left=0.20, right=0.88)

# رسم
sns.heatmap(
    correlation_matrix,
    annot=annot_matrix,
    fmt="",
    cmap=cmap,
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=1.5,
    linecolor="white",
    xticklabels=labels,
    yticklabels=labels,
    annot_kws={"fontsize": 12, "fontweight": "bold", "color": "black"},
    cbar_kws={"shrink": 0.80},
    ax=ax,
)

# محور افقی
ax.set_xticklabels(labels, rotation=45, ha="right", rotation_mode="anchor", fontsize=12)

# محور عمودی
ax.set_yticklabels(labels, rotation=0, fontsize=12)

# فاصله برچسب‌ها
ax.tick_params(axis="x", pad=10)
ax.tick_params(axis="y", pad=10)

# عنوان
ax.set_title(
    fix_fa("ماتریس همبستگی متغیرها"), fontsize=18, fontweight="bold", pad=20
)

# نوار رنگ
cbar = ax.collections[0].colorbar
cbar.set_label(
    fix_fa("ضریب همبستگی"), fontsize=12, rotation=270, labelpad=20
)

# ذخیره و نمایش نمودار
plt.savefig("correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
"""
سوال9
"""
# وارد کردن کتابخانه‌ها 
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import arabic_reshaper
from bidi.algorithm import get_display

# از راست به چپ کردن متن فارسی
def fix_fa(text):
    reshaped_text = arabic_reshaper.reshape(str(text))
    return get_display(reshaped_text)

# خواندن دیتای مورد نیاز
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop", "final_data.csv")
df = pd.read_csv(desktop_path, low_memory=False)

city_clean = (
    df["city_slug"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df_tehran = df[
    city_clean.str.contains("tehran", na=False) |
    city_clean.eq("تهران")
].copy()

# نام فارسی محله‌ها
neighborhood_fa = {
    "saadat_abad": "سعادت‌آباد",
    "saadatabad": "سعادت‌آباد",
    "janat_abad": "جنت‌آباد",
    "janatabad": "جنت‌آباد",
    "elahiyeh": "الهیه",
    "elahieh": "الهیه",
    "simon_bolivar": "سیمون بولیوار",
    "simon-bolivar": "سیمون بولیوار",
    "shahrak_gharb": "شهرک غرب",
    "shahrak-gharb": "شهرک غرب",
    "punak": "پونک",
    "punak_shomali": "پونک شمالی",
    "punak_jonoobi": "پونک جنوبی",
    "farahzad": "فرحزاد",
    "marzdaran": "مرزداران",
    "bagh_feyz": "باغ فیض",
    "shahran": "شهران",
    "ferdows": "فردوس",
    "ekbatan": "اکباتان",
    "ariya_shahr": "آریاشهر",
    "yousef_abad": "یوسف‌آباد",
    "yusef_abad": "یوسف‌آباد",
    "mirdamad": "میرداماد",
    "vanak": "ونک",
    "qolhak": "قلهک",
    "gholhak": "قلهک",
    "farmanieh": "فرمانیه",
    "kamranieh": "کامرانیه",
    "kamraniyeh": "کامرانیه",
    "zafaraniyeh": "زعفرانیه",
    "zaferaniyeh": "زعفرانیه",
    "niavaran": "نیاوران",
    "tajrish": "تجریش",
    "velenjak": "ولنجک",
    "darakeh": "درکه",
    "qeytarieh": "قیطریه",
    "qeytarie": "قیطریه",
    "pasdaran": "پاسداران",
    "tehranpars": "تهرانپارس",
    "narmak": "نارمک",
    "lavisan": "لویزان",
    "hakimiyeh": "حکیمیه",
    "shamsabad": "شمس‌آباد",
    "sadeghieh": "صادقیه",
    "sadeghiyeh": "صادقیه",
    "gisha": "گیشا",
    "kuy_nasr": "کوی نصر",
    "moniriyeh": "منیریه",
    "jomhouri": "جمهوری",
    "yaftabad": "یافت‌آباد",
    "chitgar": "چیتگر",
    "shahrak_azadi": "شهرک آزادی",
    "shahrak_baqeri": "شهرک باقری",
    "aghdasiyeh": "اقدسیه",
    "mahmodiyeh": "محمودیه",
    "mahmoodieh": "محمودیه",
    "fereshteh": "فرشته",
    "jordaan": "جردن",
    "jordan": "جردن",
    "darrous": "دروس",
    "darus": "دروس",
    "ekhtiyariyeh": "اختیاریه",
    "niayesh": "نیایش",
    "zargandeh": "زرگنده",
    "abbas_abad": "عباس‌آباد",
    "seyed_khandan": "سیدخندان",
    "motahari": "مطهری",
    "fatemi": "فاطمی",
    "amiriyeh": "امیریه",
    "nazem_abad": "نظام‌آباد",
    "nizam_abad": "نظام‌آباد",
    "tehran_no": "تهران‌نو",
    "tehran_nou": "تهران‌نو",
    "tehran_pars": "تهرانپارس",
    "nazi_abad": "نازی‌آباد",
    "joobsh": "جوب‌شیر",
}

# تمیزکاری نام محله‌ها
df_tehran["neighborhood_clean"] = (
    df_tehran["neighborhood_slug"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace("-", "_", regex=False)
)

# مپ کردن نام‌های فارسی محله‌ها
df_tehran["neighborhood_fa"] = df_tehran["neighborhood_clean"].map(neighborhood_fa)

# تغییر در فیچرها: به‌روزرسانی اسامی ستون‌های جدید استخراج‌شده
features = {
    "has_balcony": {
        "title": "درصد خانه‌های دارای بالکن در محله‌های تهران",
        "color": "#93C078"
    },
    "has_elevator": {
        "title": "درصد خانه‌های دارای آسانسور در محله‌های تهران",
        "color": "#EC6DA2"
    },
    "is_security_guard_house_tehran": {  # ستون جدید استخراج شده برای نگهبانی/سرایداری
        "title": "درصد خانه‌های دارای نگهبانی/سرایداری در محله‌های تهران",
        "color": "#B5A5DA"
    },
    "is_barbecue_house_tehran": {  # ستون جدید استخراج شده برای باربیکیو
        "title": "درصد خانه‌های دارای باربیکیو در محله‌های تهران",
        "color": "#E8BC5A"
    },
    "is_pool_house_tehran": {  # ستون جدید استخراج شده برای استخر
        "title": "درصد خانه‌های دارای استخر در محله‌های تهران",
        "color": "#7CDBBE"
    }
}

# تابع تبدیل مقادیر متنی و متغیرها به Boolean
def convert_feature(series):
    true_values = {
        True, 1, "1", "1.0",
        "true", "True", "TRUE",
        "yes", "Yes", "YES",
        "دارد", "دارد "
    }

    false_values = {
        False, 0, "0", "0.0",
        "false", "False", "FALSE",
        "no", "No", "NO",
        "ندارد", "ندارد "
    }

    result = pd.Series(np.nan, index=series.index)
    result[series.isin(true_values)] = True
    result[series.isin(false_values)] = False
    return result

# رسم نمودارها
for feature, info in features.items():
    if feature not in df_tehran.columns:
        continue

    temp = df_tehran[["neighborhood_fa", feature]].copy()
    temp[feature] = convert_feature(temp[feature])

    temp = temp.dropna(subset=["neighborhood_fa"])
    temp_known = temp.dropna(subset=[feature])

    result = (
        temp_known
        .groupby("neighborhood_fa")[feature]
        .agg(
            total_known="count",
            has_feature="sum"
        )
        .reset_index()
    )

    result["percentage"] = (
        result["has_feature"] /
        result["total_known"] * 100
    )

    result = result.sort_values("percentage", ascending=True)

    plt.figure(figsize=(14, 12))

    plt.barh(
        result["neighborhood_fa"].apply(fix_fa),
        result["percentage"],
        color=info["color"]
    )

    plt.title(
        fix_fa(info["title"]),
        fontsize=18,
        fontweight="bold",
        pad=15
    )

    plt.xlabel(
        fix_fa("درصد خانه‌ها (%)"),
        fontsize=13
    )

    plt.ylabel(
        fix_fa("محله"),
        fontsize=13
    )

    max_val = result["percentage"].max() if len(result) > 0 else 100
    plt.xlim(0, max(max_val * 1.15, 10))

    plt.grid(
        axis="x",
        alpha=0.2
    )

    for bar, value in zip(plt.gca().patches, result["percentage"]):
        plt.text(
            value + (max_val * 0.01 if max_val > 0 else 0.5),
            bar.get_y() + bar.get_height() / 2,
            f"{value:.1f}%",
            va="center",
            fontsize=9
        )

    plt.tight_layout()
    plt.show()